In [1]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image

# Reproducibility (score-neutral; helps stability across runs)
torch.manual_seed(42)
np.random.seed(42)



In [2]:
# Data paths (keep consistent with Kaggle)
directory = "/kaggle/input/petfinder-pawpularity-score"
train_df = pd.read_csv(os.path.join(directory, "train.csv"))
test_df = pd.read_csv(os.path.join(directory, "test.csv"))

print("Train samples:", len(train_df))
print("Test samples:", len(test_df))



Train samples: 8920
Test samples: 992


In [3]:
# Fix: use a proper PyTorch Dataset that returns a single sample by index.
# Core logic preserved: CLIP image embeddings will be extracted and fed into SVR.


class PetfinderCLIPDataset(Dataset):
    def __init__(self, df: pd.DataFrame, directory: str, processor, test: bool = False):
        self.df = df.reset_index(drop=True)
        self.directory = directory
        self.processor = processor
        self.test = test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        split = "test" if self.test else "train"
        img_id = self.df.loc[idx, "Id"]
        img_path = os.path.join(self.directory, split, f"{img_id}.jpg")

        # Ensure RGB for CLIP
        image = Image.open(img_path).convert("RGB")
        enc = self.processor(images=image, return_tensors="pt")

        # Squeeze batch dimension so DataLoader stacks correctly
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if not self.test:
            item["labels"] = torch.tensor(
                self.df.loc[idx, "Pawpularity"], dtype=torch.float32
            )
        return item


def collate_fn(batch):
    # Stack dict tensors; labels optional
    out = {}
    keys = batch[0].keys()
    for k in keys:
        out[k] = torch.stack([b[k] for b in batch], dim=0)
    return out




In [4]:
# Fix: CLIP loading error from the given local path.
# Minimal change: try the provided local path if it exists; otherwise fall back to a standard HF model id.
# This keeps core logic (CLIP embeddings) intact and ensures the notebook runs without the extra dataset.
from transformers import CLIPModel, CLIPProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"

local_model_path = "/kaggle/input/clip-vit/pytorch/b-32-laion2b-s34b-b79k/1"
fallback_model_id = "openai/clip-vit-base-patch32"

model_id_to_use = (
    local_model_path if os.path.exists(local_model_path) else fallback_model_id
)
print("Loading CLIP from:", model_id_to_use)

model_clip = CLIPModel.from_pretrained(model_id_to_use)
processor_clip = CLIPProcessor.from_pretrained(model_id_to_use)

model_clip = model_clip.to(device)
model_clip.eval()



2026-05-14 14:53:40.779799: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778770420.792895      55 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778770420.796861      55 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-14 14:53:40.812281: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

In [5]:
# DataLoaders
train_dataset = PetfinderCLIPDataset(train_df, directory, processor_clip, test=False)
test_dataset = PetfinderCLIPDataset(test_df, directory, processor_clip, test=True)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_fn,
)
test_dataloader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_fn,
)




In [6]:
# Feature extraction (fix: move dict tensors to device correctly; don't call .to() on dict)
def extract_clip_features(dataloader, model, device, has_labels: bool):
    feats = []
    labels = [] if has_labels else None

    for batch in tqdm(dataloader, total=len(dataloader)):
        batch_inputs = {
            k: v.to(device, non_blocking=True)
            for k, v in batch.items()
            if k != "labels"
        }
        with torch.no_grad():
            x = model.get_image_features(**batch_inputs)  # (B, D)
        feats.append(x.detach().cpu().numpy())

        if has_labels:
            labels.append(batch["labels"].detach().cpu().numpy())

    feats = np.concatenate(feats, axis=0)
    if has_labels:
        labels = np.concatenate(labels, axis=0)
        return feats, labels
    return feats


X, y = extract_clip_features(train_dataloader, model_clip, device, has_labels=True)
X_test = extract_clip_features(test_dataloader, model_clip, device, has_labels=False)

print("final X:", X.shape, "y:", y.shape)
print("final X_test:", X_test.shape)




100%|██████████| 16/16 [00:06<00:00,  2.60it/s]

final X: (8920, 512) y: (8920,)
final X_test: (992, 512)


In [7]:
# Scaling (same intention as original; fix: avoid empty arrays and ensure scaler is fitted)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(np.vstack([X, X_test]))
X_scaled = scaler.transform(X)
X_test_scaled = scaler.transform(X_test)

print("X_scaled:", X_scaled.shape, "X_test_scaled:", X_test_scaled.shape)



X_scaled: (8920, 512) X_test_scaled: (992, 512)


In [8]:
# Fix: cuML SVR cannot import due to CUDA driver mismatch; use sklearn SVR (same algorithmic idea).
from sklearn.svm import SVR

# Keep hyperparameters as close as possible to the original intent.
reg = SVR(C=16.0, kernel="rbf", degree=3, max_iter=400000)
reg.fit(X_scaled, y)

print("Predicted values (train head):", reg.predict(X_scaled[:10, :]))
print("True values (train head):", y[:10])



Predicted values (train head): [54.89993287 75.06079329 27.09992446 44.946591   38.75995292 29.37705931
 35.97908394 35.90002744 31.73942316 48.6470225 ]
True values (train head): [ 55. 100.  27.  51.  48.  32.  25.  36.  34.  60.]


In [9]:
# Predict on test
y_pred = reg.predict(X_test_scaled)

# Safety: ensure predictions are numeric and within plausible range for Pawpularity.
# This is score-stabilizing and prevents invalid outputs.
y_pred = np.asarray(y_pred, dtype=np.float32)
y_pred = np.clip(y_pred, 0.0, 100.0)

print("Test predictions:", y_pred[:10], "shape:", y_pred.shape)



Test predictions: [38.578094 38.76181  38.470867 31.4895   27.369757 36.581436 19.964708
 38.251328 39.509125 39.79636 ] shape: (992,)


In [10]:
# Submission (fix: correct header/columns, ensure .csv suffix)
submit = pd.DataFrame({"Id": test_df["Id"].values, "Pawpularity": y_pred})
submit.to_csv("submission.csv", index=False)

print("Wrote submission.csv with shape:", submit.shape)



Wrote submission.csv with shape: (992, 2)


In [11]:
submit.head()

,Id,Pawpularity
0,ee51b99832f1ba868f646df93d2b6b81,38.578094
1,caddfb3f8bff9c4b95dbe022018eea21,38.761810
2,582eeabd4a448a53ebb79995888a4b0b,38.470867
3,afc1ad7f0c5eea880759d09e77f7deee,31.489500
4,d5bdf3446e86ce4ec67ce7a00f1cccc2,27.369757
